In [ ]:
import os, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Conv1D, MaxPooling1D, Dropout,
    Flatten, Activation, Lambda, Dot, ELU
)
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# Paths
CLOSED_DATA_PATH = ''   # closed-world directory
OPEN_DATA_PATH   = ''   # open-world (unmonitored) directory

# Sequence
INPUT_LEN = 5000   # sequence length (pad/truncate)
EMB_SIZE  = 64     # embedding size
ALPHA     = 0.1    # triplet margin

# Training
BATCH_SIZE = 128
N_EPOCHS   = 100

# Open-world sample counts
N_UNKNOWN_TRAIN = 150
N_UNKNOWN_TEST  = 10000

# Split ratio
TRAIN_RATIO = 0.8

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
def load_trace_direction(filepath, max_len=INPUT_LEN):
    directions = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            try:
                size = float(parts[1])
                directions.append(1.0 if size > 0 else -1.0)
            except ValueError:
                continue
    # truncate
    directions = directions[:max_len]
    # pad
    if len(directions) < max_len:
        directions += [0.0] * (max_len - len(directions))
    return np.array(directions, dtype=np.float32)


def load_dataset(data_path, label_override=None, max_len=INPUT_LEN, max_per_class=None):
    traces, labels = [], []
    subdirs = sorted(os.listdir(data_path))
    for subdir in subdirs:
        subdir_path = os.path.join(data_path, subdir)
        if not os.path.isdir(subdir_path):
            continue
        label = label_override if label_override is not None else subdir
        count = 0
        for fname in sorted(os.listdir(subdir_path)):
            if not fname.endswith('.txt'):
                continue
            if max_per_class is not None and count >= max_per_class:
                break
            fpath = os.path.join(subdir_path, fname)
            try:
                trace = load_trace_direction(fpath, max_len)
                traces.append(trace)
                labels.append(label)
                count += 1
            except Exception as e:
                print(f"[WARN] skip {fpath}: {e}")
    return np.array(traces, dtype=np.float32), np.array(labels)


In [ ]:
MAX_PER_CLASS = 200  # giới hạn số sample mỗi label

print("=== Loading closed-world data ===")
closed_traces, closed_labels = load_dataset(CLOSED_DATA_PATH, max_per_class=MAX_PER_CLASS)
print(f"Closed: {closed_traces.shape},  classes: {np.unique(closed_labels).shape[0]}")

print("=== Loading open-world (unknown) data ===")
open_traces, open_labels = load_dataset(OPEN_DATA_PATH, label_override='unknown', max_per_class=MAX_PER_CLASS)
print(f"Open  : {open_traces.shape}")

# Split closed-world
c_tr_x, c_te_x, c_tr_y, c_te_y = train_test_split(
    closed_traces, closed_labels,
    train_size=TRAIN_RATIO, random_state=SEED, stratify=closed_labels
)

# Split unknown
idx_all = np.arange(len(open_traces))
np.random.shuffle(idx_all)
unk_train_idx = idx_all[:N_UNKNOWN_TRAIN]
unk_test_idx  = idx_all[N_UNKNOWN_TRAIN:N_UNKNOWN_TRAIN + N_UNKNOWN_TEST]
unk_tr_x = open_traces[unk_train_idx]
unk_te_x = open_traces[unk_test_idx]
unk_tr_y = open_labels[unk_train_idx]
unk_te_y = open_labels[unk_test_idx]

# Combine for classifier training / testing
clf_train_x = np.concatenate([c_tr_x, unk_tr_x], axis=0)
clf_train_y = np.concatenate([c_tr_y, unk_tr_y], axis=0)
clf_test_x  = np.concatenate([c_te_x, unk_te_x], axis=0)
clf_test_y  = np.concatenate([c_te_y, unk_te_y], axis=0)

# Encode labels
le = LabelEncoder()
le.fit(clf_train_y)
clf_train_y_enc = le.transform(clf_train_y)
clf_test_y_enc  = le.transform(clf_test_y)
num_classes = len(le.classes_)
print(f"Total classes (closed + unknown): {num_classes}")
print(f"Classes: {le.classes_}")

# Reshape for Conv1D: (N, L, 1)
c_tr_x_3d   = c_tr_x[:, :, np.newaxis]
clf_tr_x_3d = clf_train_x[:, :, np.newaxis]
clf_te_x_3d = clf_test_x[:, :, np.newaxis]

In [ ]:
def DF(input_shape=None, emb_size=None):
    # -----------------Entry flow -----------------
    input_data = Input(shape=input_shape)

    filter_num = ['None', 32, 64, 128, 256]
    kernel_size = ['None', 8, 8, 8, 8]
    conv_stride_size = ['None', 1, 1, 1, 1]
    pool_stride_size = ['None', 4, 4, 4, 4]
    pool_size = ['None', 8, 8, 8, 8]

    model = Conv1D(filters=filter_num[1], kernel_size=kernel_size[1],
                   strides=conv_stride_size[1], padding='same', name='block1_conv1')(input_data)
    model = ELU(alpha=1.0, name='block1_adv_act1')(model)
    model = Conv1D(filters=filter_num[1], kernel_size=kernel_size[1],
                   strides=conv_stride_size[1], padding='same', name='block1_conv2')(model)
    model = ELU(alpha=1.0, name='block1_adv_act2')(model)
    model = MaxPooling1D(pool_size=pool_size[1], strides=pool_stride_size[1],
                         padding='same', name='block1_pool')(model)
    model = Dropout(0.1, name='block1_dropout')(model)

    model = Conv1D(filters=filter_num[2], kernel_size=kernel_size[2],
                   strides=conv_stride_size[2], padding='same', name='block2_conv1')(model)
    model = Activation('relu', name='block2_act1')(model)
    model = Conv1D(filters=filter_num[2], kernel_size=kernel_size[2],
                   strides=conv_stride_size[2], padding='same', name='block2_conv2')(model)
    model = Activation('relu', name='block2_act2')(model)
    model = MaxPooling1D(pool_size=pool_size[2], strides=pool_stride_size[3],
                         padding='same', name='block2_pool')(model)
    model = Dropout(0.1, name='block2_dropout')(model)

    model = Conv1D(filters=filter_num[3], kernel_size=kernel_size[3],
                   strides=conv_stride_size[3], padding='same', name='block3_conv1')(model)
    model = Activation('relu', name='block3_act1')(model)
    model = Conv1D(filters=filter_num[3], kernel_size=kernel_size[3],
                   strides=conv_stride_size[3], padding='same', name='block3_conv2')(model)
    model = Activation('relu', name='block3_act2')(model)
    model = MaxPooling1D(pool_size=pool_size[3], strides=pool_stride_size[3],
                         padding='same', name='block3_pool')(model)
    model = Dropout(0.1, name='block3_dropout')(model)

    model = Conv1D(filters=filter_num[4], kernel_size=kernel_size[4],
                   strides=conv_stride_size[4], padding='same', name='block4_conv1')(model)
    model = Activation('relu', name='block4_act1')(model)
    model = Conv1D(filters=filter_num[4], kernel_size=kernel_size[4],
                   strides=conv_stride_size[4], padding='same', name='block4_conv2')(model)
    model = Activation('relu', name='block4_act2')(model)
    model = MaxPooling1D(pool_size=pool_size[4], strides=pool_stride_size[4],
                         padding='same', name='block4_pool')(model)

    output = Flatten()(model)

    dense_layer = Dense(emb_size, name='FeaturesVec')(output)
    shared_conv2 = Model(inputs=input_data, outputs=dense_layer)
    return shared_conv2


In [ ]:
# ─────────────────────────────────────────────
# 4.  TRIPLET LOSS (Phase 1) – train embedding
# ─────────────────────────────────────────────

def build_pos_pairs_for_id(classid, classid_to_ids, max_pairs_per_class=200):
    traces = classid_to_ids[classid]
    pos_pairs = [(traces[i], traces[j])
                 for i in range(len(traces))
                 for j in range(i+1, len(traces))]
    random.shuffle(pos_pairs)
    return pos_pairs[:max_pairs_per_class]  # ← giới hạn lại

def build_positive_pairs(class_id_range, classid_to_ids):
    listX1, listX2 = [], []
    for cid in class_id_range:
        for (a, p) in build_pos_pairs_for_id(cid, classid_to_ids, max_pairs_per_class=200):
            listX1.append(a); listX2.append(p)
    perm = np.random.permutation(len(listX1))
    return np.array(listX1)[perm], np.array(listX2)[perm]

def build_similarities(conv, all_traces, chunk_size=2000):
    embs = conv.predict(all_traces, batch_size=256, verbose=0)
    embs = embs / (np.linalg.norm(embs, axis=-1, keepdims=True) + 1e-8)
    embs = embs.astype(np.float16)  # float16: 1.5GB → 750MB

    N = len(embs)
    sim_matrix = np.empty((N, N), dtype=np.float16)
    for i in range(0, N, chunk_size):
        end = min(i + chunk_size, N)
        sim_matrix[i:end] = np.dot(embs[i:end], embs.T)

    return sim_matrix  # CPU RAM, không chiếm GPU VRAM

def intersect(a, b):
    return list(set(a) & set(b))

def build_negatives(anc_idxs, pos_idxs, similarities, neg_imgs_idx, id_to_classid, num_retries=50):
    if similarities is None:
        return random.sample(list(neg_imgs_idx), len(anc_idxs))

    neg_imgs_arr = np.array(list(neg_imgs_idx))
    # Precompute class array để so sánh nhanh
    classid_arr = np.array([id_to_classid.get(i, -1) for i in neg_imgs_arr])

    final_neg = []
    for anc_idx, pos_idx in zip(anc_idxs, pos_idxs):
        anchor_class = id_to_classid[anc_idx]
        sim = similarities[anc_idx, pos_idx]

        # Vectorized: tìm tất cả candidate thỏa điều kiện cùng lúc
        mask_sim      = (similarities[anc_idx, neg_imgs_arr] + float(ALPHA)) > sim
        mask_diff_cls = classid_arr != anchor_class
        candidates    = neg_imgs_arr[mask_sim & mask_diff_cls]

        if len(candidates) > 0:
            final_neg.append(int(np.random.choice(candidates)))
        else:
            final_neg.append(int(np.random.choice(neg_imgs_arr)))

    return final_neg


class SemiHardTripletGenerator:
    def __init__(self, Xa, Xp, batch_size, all_traces, neg_idx, id_to_classid, conv=None):
        self.Xa, self.Xp = Xa, Xp
        self.batch_size = batch_size
        self.traces = all_traces
        self.neg_idx = neg_idx
        self.id_to_classid = id_to_classid
        self.cur = 0
        self.num_samples = len(Xa)
        self.similarities = build_similarities(conv, all_traces) if conv else None

    def __iter__(self):
        return self

    def __next__(self):
        self.cur += self.batch_size
        if self.cur >= self.num_samples:
            self.cur = 0
        a_idx = self.Xa[self.cur:self.cur + self.batch_size]
        p_idx = self.Xp[self.cur:self.cur + self.batch_size]
        n_idx = build_negatives(a_idx, p_idx, self.similarities,
                                self.neg_idx, self.id_to_classid)
        return (
            (self.traces[a_idx], self.traces[p_idx], self.traces[np.array(n_idx)]),
            np.zeros(len(a_idx))
        )

    def as_tf_dataset(self, steps):
        def gen():
            for _ in range(steps):
                yield self.__next__()
        sig = (
            (
                tf.TensorSpec(shape=(None, INPUT_LEN, 1), dtype=tf.float32),
                tf.TensorSpec(shape=(None, INPUT_LEN, 1), dtype=tf.float32),
                tf.TensorSpec(shape=(None, INPUT_LEN, 1), dtype=tf.float32),
            ),
            tf.TensorSpec(shape=(None,), dtype=tf.float32)
        )
        return tf.data.Dataset.from_generator(gen, output_signature=sig)

    # Trong SemiHardTripletGenerator, thêm method này:
    def as_infinite_dataset(self):
        """Dataset vô hạn, không tạo lại mỗi epoch."""
        def gen():
            while True:
                yield self.__next__()
        sig = (
            (
                tf.TensorSpec(shape=(None, INPUT_LEN, 1), dtype=tf.float32),
                tf.TensorSpec(shape=(None, INPUT_LEN, 1), dtype=tf.float32),
                tf.TensorSpec(shape=(None, INPUT_LEN, 1), dtype=tf.float32),
            ),
            tf.TensorSpec(shape=(None,), dtype=tf.float32)
        )
        return tf.data.Dataset.from_generator(gen, output_signature=sig)


# ── Build encoder + TripletModel ──────────────
encoder =  DF(input_shape=(INPUT_LEN, 1), emb_size=EMB_SIZE)
encoder.summary()


class TripletModel(tf.keras.Model):
    """Wrapper dùng custom train_step để tránh lỗi Keras 3 với Lambda loss."""
    def __init__(self, encoder, alpha=0.1):
        super().__init__()
        self.encoder = encoder
        self.alpha = float(alpha)
        self.loss_tracker = tf.keras.metrics.Mean(name='loss')

    @property
    def metrics(self):
        return [self.loss_tracker]

    def call(self, inputs, training=False):
        anchor, positive, negative = inputs
        return (
            self.encoder(anchor,   training=training),
            self.encoder(positive, training=training),
            self.encoder(negative, training=training),
        )

    def _triplet_loss(self, a_emb, p_emb, n_emb):
        a = tf.math.l2_normalize(a_emb, axis=-1)
        p = tf.math.l2_normalize(p_emb, axis=-1)
        n = tf.math.l2_normalize(n_emb, axis=-1)
        pos_sim = tf.reduce_sum(a * p, axis=-1)
        neg_sim = tf.reduce_sum(a * n, axis=-1)
        return tf.reduce_mean(tf.maximum(0.0, neg_sim - pos_sim + self.alpha))

    def train_step(self, data):
        inputs, _ = data
        with tf.GradientTape() as tape:
            a_emb, p_emb, n_emb = self(inputs, training=True)
            loss = self._triplet_loss(a_emb, p_emb, n_emb)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.loss_tracker.update_state(loss)
        return {'loss': self.loss_tracker.result()}

    def test_step(self, data):
        inputs, _ = data
        a_emb, p_emb, n_emb = self(inputs, training=False)
        loss = self._triplet_loss(a_emb, p_emb, n_emb)
        self.loss_tracker.update_state(loss)
        return {'loss': self.loss_tracker.result()}


triplet_model = TripletModel(encoder, alpha=ALPHA)
triplet_model.compile(
    optimizer=tf.keras.optimizers.SGD(
        learning_rate=0.001,
        momentum=0.9,
        nesterov=True
    ),
    run_eagerly=True
)

# ── Build triplet index (closed-world only) ──
print("=== Building triplet index (closed-world only) ===")

# Dùng trực tiếp c_tr_y (đã split) thay vì rebuild từ filesystem
unique_train_classes = sorted(set(c_tr_y))
cid_map = {cls: i for i, cls in enumerate(unique_train_classes)}

# Map từ local index (0 → len(c_tr_x)-1) sang class id
classid_to_ids_trip = {i: [] for i in range(len(unique_train_classes))}
for local_i, label in enumerate(c_tr_y):
    classid_to_ids_trip[cid_map[label]].append(local_i)

id_to_classid_trip = {}
for cid, ids in classid_to_ids_trip.items():
    for idx in ids:
        id_to_classid_trip[idx] = cid

Xa_train, Xp_train = build_positive_pairs(
    range(len(unique_train_classes)), classid_to_ids_trip
)
all_train_idx = list(set(Xa_train.tolist()) | set(Xp_train.tolist()))

print(f"Triplet pairs  : {len(Xa_train)}")
print(f"Train traces   : {len(all_train_idx)}")
print(f"c_tr_x_3d size : {len(c_tr_x_3d)}")
print(f"Max index      : {max(all_train_idx)}  (must be < {len(c_tr_x_3d)})")

In [ ]:
print("\n=== Phase 1: Training triplet embedding ===")
os.makedirs('models', exist_ok=True)

# Trong cell 5, thay dòng này:
steps_per_epoch = len(Xa_train) // BATCH_SIZE  # = 30489, quá lớn

# Thành:
steps_per_epoch = min(len(Xa_train) // BATCH_SIZE, 500)  # ~64K samples/epoch là đủ

gen = SemiHardTripletGenerator(Xa_train, Xp_train, BATCH_SIZE,
                                c_tr_x_3d, all_train_idx, id_to_classid_trip, conv=None)

for epoch in range(N_EPOCHS):
    print(f"\n[Epoch {epoch+1}/{N_EPOCHS}]")

    # Tạo dataset từ generator hiện tại
    ds = gen.as_infinite_dataset()

    triplet_model.fit(ds, steps_per_epoch=steps_per_epoch, epochs=1, verbose=1)

    # Rebuild generator với similarities mới (semi-hard mining)
    # Chỉ rebuild sau epoch 1 trở đi để tránh lãng phí epoch đầu
    gen = SemiHardTripletGenerator(Xa_train, Xp_train, BATCH_SIZE,
                                    c_tr_x_3d, all_train_idx, id_to_classid_trip, conv=encoder)

encoder.save('models/DF_encoder.h5')
print("Encoder saved.")

In [ ]:
print("\n=== Extracting embeddings ===")
emb_train = encoder.predict(clf_tr_x_3d, batch_size=64, verbose=1)
emb_test  = encoder.predict(clf_te_x_3d, batch_size=64, verbose=1)

# L2 normalize
emb_train = emb_train / (np.linalg.norm(emb_train, axis=1, keepdims=True) + 1e-8)
emb_test  = emb_test  / (np.linalg.norm(emb_test,  axis=1, keepdims=True) + 1e-8)

In [ ]:
from collections import defaultdict

# ── CONFIG few-shot ──────────────────────────
N_SHOT_LIST   = [1, 5, 10, 20]   
THRESHOLD     = 0.5              
N_REPEAT      = 10               

# ── Helpers ──────────────────────────────────

def build_class_pool(emb_array, label_array):
    """
    Gom embedding theo class.
    Trả về dict: class_name → list of embeddings (np.array)
    """
    pool = defaultdict(list)
    for emb, lbl in zip(emb_array, label_array):
        pool[lbl].append(emb)
    return pool


def sample_support_query(pool, n_shot, rng):
    signatures = {}
    query_embs = []

    for cls, embs in pool.items():
        embs = np.array(embs)
        if len(embs) <= n_shot:
            signatures[cls] = embs.mean(axis=0)
            continue
        idx = np.arange(len(embs))
        rng.shuffle(idx)
        support_idx = idx[:n_shot]
        query_idx   = idx[n_shot:]
        signatures[cls] = embs[support_idx].mean(axis=0)   # N-MEV
        for qi in query_idx:
            query_embs.append((embs[qi], cls))

    return signatures, query_embs


def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


def knn_predict(query_emb, signatures, threshold):
    best_cls, best_sim = None, -1.0
    for cls, sig in signatures.items():
        sim = cosine_sim(query_emb, sig)
        if sim > best_sim:
            best_sim, best_cls = sim, cls
    if best_sim < threshold:
        return 'unknown', best_sim
    return best_cls, best_sim


def evaluate_once(pool_closed, pool_unknown, n_shot, threshold, rng):
    signatures, closed_queries = sample_support_query(pool_closed, n_shot, rng)

    # --- Closed-world queries ---
    y_true_c, y_pred_c = [], []
    for (emb, true_cls) in closed_queries:
        pred, _ = knn_predict(emb, signatures, threshold)
        y_true_c.append(true_cls)
        y_pred_c.append(pred)

    correct_closed = sum(t == p for t, p in zip(y_true_c, y_pred_c))
    acc_closed = correct_closed / len(y_true_c) if y_true_c else 0.0

    # --- Unknown queries ---
    acc_unknown = None
    if pool_unknown:
        unk_embs = np.array(pool_unknown)   
        correct_unk = 0
        for emb in unk_embs:
            pred, _ = knn_predict(emb, signatures, threshold)
            if pred == 'unknown':
                correct_unk += 1
        acc_unknown = correct_unk / len(unk_embs)

    return acc_closed, acc_unknown

In [ ]:
print("\n=== Extracting embeddings ===")

all_closed_x3d = c_te_x[:, :, np.newaxis]   
emb_closed = encoder.predict(all_closed_x3d, batch_size=64, verbose=1)
emb_closed = emb_closed / (np.linalg.norm(emb_closed, axis=1, keepdims=True) + 1e-8)

pool_closed = build_class_pool(emb_closed, c_te_y)   

pool_unknown = []
if len(open_traces) > 0:
    all_open_x3d = open_traces[:, :, np.newaxis]
    emb_open = encoder.predict(all_open_x3d, batch_size=64, verbose=1)
    emb_open = emb_open / (np.linalg.norm(emb_open, axis=1, keepdims=True) + 1e-8)
    pool_unknown = list(emb_open)

In [ ]:
print("\n=== Few-shot kNN Experiment ===")
print(f"Threshold = {THRESHOLD}, N_REPEAT = {N_REPEAT}\n")

results = {}

for n_shot in N_SHOT_LIST:
    acc_closed_list  = []
    acc_unknown_list = []

    for trial in range(N_REPEAT):
        rng = np.random.default_rng(SEED + trial)
        acc_c, acc_u = evaluate_once(
            pool_closed  = pool_closed,
            pool_unknown = pool_unknown,
            n_shot       = n_shot,
            threshold    = THRESHOLD,
            rng          = rng
        )
        acc_closed_list.append(acc_c)
        if acc_u is not None:
            acc_unknown_list.append(acc_u)

    results[n_shot] = {
        'closed' : acc_closed_list,
        'unknown': acc_unknown_list
    }

    mean_c = np.mean(acc_closed_list)
    std_c  = np.std(acc_closed_list)
    print(f"[n_shot={n_shot:2d}]  Closed-world Acc : {mean_c:.4f} ± {std_c:.4f}")
    if acc_unknown_list:
        mean_u = np.mean(acc_unknown_list)
        std_u  = np.std(acc_unknown_list)
        print(f"           Unknown Detection: {mean_u:.4f} ± {std_u:.4f}")
    print()


def collect_predictions_closed(pool_closed, n_shot, n_repeat, seed):
    all_true, all_pred, all_scores = [], [], []
    for trial in range(n_repeat):
        rng = np.random.default_rng(seed + trial)
        signatures, queries = sample_support_query(pool_closed, n_shot, rng)
        for (emb, true_cls) in queries:
            pred, sim = knn_predict(emb, signatures, threshold=THRESHOLD)  # ← đổi 0.0 → THRESHOLD
            all_true.append(true_cls)
            all_pred.append(pred)
            all_scores.append(sim)
    return all_true, all_pred, all_scores


def collect_predictions_open(pool_closed, pool_unknown, n_shot, n_repeat, seed):
    """
    Gom y_true_binary / sim_score qua N_REPEAT lần
    cho kịch bản open-world (known=1, unknown=0).
    y_pred dùng THRESHOLD để phân loại.
    """
    all_true_cls, all_pred_cls = [], []   # nhãn đầy đủ để classification report
    all_true_bin, all_scores   = [], []   # binary + score để vẽ curve

    for trial in range(n_repeat):
        rng = np.random.default_rng(seed + trial)
        signatures, closed_queries = sample_support_query(pool_closed, n_shot, rng)

        # --- closed queries ---
        for (emb, true_cls) in closed_queries:
            pred, sim = knn_predict(emb, signatures, threshold=THRESHOLD)
            all_true_cls.append(true_cls)
            all_pred_cls.append(pred)
            all_true_bin.append(1)   # known
            all_scores.append(sim)

        # --- unknown queries ---
        for emb in pool_unknown:
            pred, sim = knn_predict(emb, signatures, threshold=THRESHOLD)
            all_true_cls.append('unknown')
            all_pred_cls.append(pred)
            all_true_bin.append(0)   # unknown
            all_scores.append(sim)

    return all_true_cls, all_pred_cls, all_true_bin, all_scores


REPORT_N_SHOT = N_SHOT_LIST[1]

print("=" * 60)
print(f"[SCENARIO 1] Closed-world | n_shot={REPORT_N_SHOT}")
print("=" * 60)

true_c, pred_c, scores_c = collect_predictions_closed(
    pool_closed, REPORT_N_SHOT, N_REPEAT, SEED
)
print(classification_report(true_c, pred_c, zero_division=0, digits=4))
print(f"Overall Accuracy: {accuracy_score(true_c, pred_c):.4f}\n")


# ── Kịch bản 2: Open-world ───────────────────
print("=" * 60)
print(f"[SCENARIO 2] Open-world   | n_shot={REPORT_N_SHOT} | threshold={THRESHOLD}")
print("=" * 60)

pred_bin_o = [0 if p == 'unknown' else 1 for p in pred_cls_o]
print("--- Binary Report (known=1 / unknown=0) ---")
print(classification_report(true_bin_o, pred_bin_o,
                             target_names=['unknown', 'known'], zero_division=0))


from sklearn.metrics import (
    precision_recall_curve, roc_curve, auc,
    average_precision_score, roc_auc_score
)

scores_arr   = np.array(scores_o,   dtype=np.float64)
true_bin_arr = np.array(true_bin_o, dtype=np.int32)

THRESHOLDS_PLOT = np.linspace(0.0, 1.0, 1000)

precisions, recalls = [], []
for thr in THRESHOLDS_PLOT:
    pred_bin_thr = (scores_arr >= thr).astype(int)
    tp = np.sum((pred_bin_thr == 1) & (true_bin_arr == 1))
    fp = np.sum((pred_bin_thr == 1) & (true_bin_arr == 0))
    fn = np.sum((pred_bin_thr == 0) & (true_bin_arr == 1))
    prec = tp / (tp + fp + 1e-12)
    rec  = tp / (tp + fn + 1e-12)
    precisions.append(prec)
    recalls.append(rec)

precisions = np.array(precisions)
recalls    = np.array(recalls)
ap_score   = average_precision_score(true_bin_arr, scores_arr)

import pandas as pd
pd.DataFrame({
    'threshold': THRESHOLDS_PLOT,
    'precision': precisions,
    'recall':    recalls
}).to_csv('pr_curve_openworld.csv', index=False)
print("Saved: pr_curve_openworld.csv")

fig_pr, ax_pr = plt.subplots(figsize=(7, 5))
ax_pr.plot(recalls, precisions, color='steelblue', lw=2,
           label=f'AP = {ap_score:.4f}')
ax_pr.axvline(x=recalls[np.argmin(np.abs(THRESHOLDS_PLOT - THRESHOLD))],
              color='gray', linestyle='--', alpha=0.6, label=f'threshold={THRESHOLD}')
ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.set_title(f'Precision-Recall Curve (Open-world, n_shot={REPORT_N_SHOT})')
ax_pr.legend()
ax_pr.grid(True, alpha=0.4)
ax_pr.set_xlim([0.0, 1.0])
ax_pr.set_ylim([0.0, 1.05])
fig_pr.tight_layout()
fig_pr.savefig('pr_curve_openworld.png', dpi=150)
plt.show()
print("Saved: pr_curve_openworld.png")

fprs, tprs = [], []
for thr in THRESHOLDS_PLOT:
    pred_bin_thr = (scores_arr >= thr).astype(int)
    tp = np.sum((pred_bin_thr == 1) & (true_bin_arr == 1))
    fp = np.sum((pred_bin_thr == 1) & (true_bin_arr == 0))
    tn = np.sum((pred_bin_thr == 0) & (true_bin_arr == 0))
    fn = np.sum((pred_bin_thr == 0) & (true_bin_arr == 1))
    fpr = fp / (fp + tn + 1e-12)
    tpr = tp / (tp + fn + 1e-12)
    fprs.append(fpr)
    tprs.append(tpr)

fprs     = np.array(fprs)
tprs     = np.array(tprs)
roc_auc  = roc_auc_score(true_bin_arr, scores_arr)

pd.DataFrame({
    'threshold': THRESHOLDS_PLOT,
    'fpr':       fprs,
    'tpr':       tprs
}).to_csv('roc_curve_openworld.csv', index=False)
print("Saved: roc_curve_openworld.csv")

fig_roc, ax_roc = plt.subplots(figsize=(7, 5))
ax_roc.plot(fprs, tprs, color='darkorange', lw=2,
            label=f'AUC = {roc_auc:.4f}')
ax_roc.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random')
thr_idx = np.argmin(np.abs(THRESHOLDS_PLOT - THRESHOLD))
ax_roc.scatter(fprs[thr_idx], tprs[thr_idx], color='red', zorder=5,
               label=f'threshold={THRESHOLD} (FPR={fprs[thr_idx]:.3f}, TPR={tprs[thr_idx]:.3f})')
ax_roc.set_xlabel('False Positive Rate')
ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title(f'ROC Curve (Open-world, n_shot={REPORT_N_SHOT})')
ax_roc.legend()
ax_roc.grid(True, alpha=0.4)
ax_roc.set_xlim([0.0, 1.0])
ax_roc.set_ylim([0.0, 1.05])
fig_roc.tight_layout()
fig_roc.savefig('roc_curve_openworld.png', dpi=150)
plt.show()
print("Saved: roc_curve_openworld.png")
print("\nDone.")